# Fine tuning, for the n=4 case

Number 2; 

COLA for continued model; and SST-2 below that 

In [1]:
import torch
import torch.nn as nn

# Needed for parallel 
from collections import OrderedDict

# For training 
from network_architecture_v2 import MyBertForSequenceClassification

# For fine tuning
from datasets import load_dataset #, load_metric
from transformers import BertTokenizer
from transformers import Trainer, TrainingArguments
import numpy as np

In [2]:
# Load dataset
dataset = load_dataset('glue', 'cola')

# I believe this is the tokenizer I used... 
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", 
                     max_length=224, truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the parallel model

This involves a bit more code

In [3]:
checkpoint_0 = torch.load('bert-save-4-continue-2/model_checkpoint_0_batch_idx=15000')
checkpoint_1 = torch.load('bert-save-4-continue-2/model_checkpoint_1_batch_idx=15000')
checkpoint_2 = torch.load('bert-save-4-continue-2/model_checkpoint_2_batch_idx=15000')
checkpoint_3 = torch.load('bert-save-4-continue-2/model_checkpoint_3_batch_idx=15000')

In [4]:
keys_0 = checkpoint_0['model_state_dict'].keys()
keys_1 = checkpoint_1['model_state_dict'].keys()
keys_2 = checkpoint_2['model_state_dict'].keys()
keys_3 = checkpoint_3['model_state_dict'].keys()

In [5]:
# Ugh, this is so dumb
new_dict = OrderedDict()
keys_0 = checkpoint_0['model_state_dict'].keys()
counter = 0
for key in keys_0:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        if int(split[2]) > counter:
            counter = int(split[2])
            
        split.insert(3, 'layer')
        new_key = '.'.join(split[1:])
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
    else:
        new_key = key
        if 'close_nsp' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_nsp'
            new_key = '.'.join(split)
        if 'close_mlm' in key:
            # print(key)
            split = key.split('.')
            split[0] = 'close_nn_mlm'
            new_key = '.'.join(split)
        
        new_dict[new_key] = checkpoint_0['model_state_dict'][key]
        
print(counter)

# Now for the remaining parts? 
keys_1 = checkpoint_1['model_state_dict'].keys()
new_counter = 0
for key in keys_1:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')

        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_1['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_1['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_2 = checkpoint_2['model_state_dict'].keys()
for key in keys_2:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_2['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_2['model_state_dict'][key]

print(new_counter)
counter = new_counter
new_counter = 0
keys_3 = checkpoint_3['model_state_dict'].keys()
for key in keys_3:
    if 'parallel_nn' in key:
        split = key.split('.')
        split[1] = 'serial_nn'
        split[2] = str(int(split[2]) + counter + 1)
        split.insert(3, 'layer')
        if int(split[2]) > new_counter:
            new_counter = int(split[2])
        new_key = '.'.join(split[1:])
        # print(key, new_key)
        new_dict[new_key] = checkpoint_3['model_state_dict'][key]
    else:
        new_dict[key] = checkpoint_3['model_state_dict'][key]

32
64
96


In [32]:
model_parallel = torch.load('serialnet_bert_128', weights_only=False)
# model_parallel.load_state_dict(new_dict)

In [33]:
training_parallel = MyBertForSequenceClassification(model_parallel)

# With weights loaded, go ahead and train

In [8]:
training_args = TrainingArguments(
    output_dir='./results_2',  # Directory to save model checkpoints and outputs
    num_train_epochs=3,  # Number of epochs for training
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=8,  # Batch size for evaluation
    learning_rate=3e-5,  # Learning rate for optimizer
    adam_beta1=0.9,  # Beta1 parameter for Adam optimizer
    adam_beta2=0.98,  # Beta2 parameter for Adam optimizer (fixed typo from 0.988)
    adam_epsilon=1e-6,  # Epsilon parameter for Adam optimizer
    dataloader_drop_last=True,  # Drop the last incomplete batch if dataset size is not divisible by batch size
    warmup_steps=100,  # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,  # Weight decay for regularization (adjusted from 1e-4 to a more typical value for transformers)
    logging_dir='./logs-2',  # Directory for logging
    logging_steps=50,  # Log every 10 steps
    eval_strategy="epoch",  # Perform evaluation at the end of each epoch
    save_strategy="epoch",  # Save model checkpoints at the end of each epoch
    save_total_limit=3,  # Limit the number of saved checkpoints to avoid excessive disk usage
    load_best_model_at_end=True,  # Load the best model (based on evaluation metric) at the end of training
    metric_for_best_model="eval_loss",  # Metric to determine the best model (adjust if using a custom metric)
)

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}

In [10]:
# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.639700,0.619615,0.691346
2,0.607400,0.622410,0.691346
3,0.633100,0.623264,0.691346


TrainOutput(global_step=3204, training_loss=0.6158725692984763, metrics={'train_runtime': 2983.8794, 'train_samples_per_second': 8.597, 'train_steps_per_second': 1.074, 'total_flos': 0.0, 'train_loss': 0.6158725692984763, 'epoch': 3.0})

# Above was COLA, moving to SST2

## WTF COLA's testset is all 0s. just use .evaluate() 

In [34]:
# Load SST-2 dataset
dataset = load_dataset('glue', 'sst2')

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],  # SST-2 uses a single sentence as input
        padding="max_length",  # Pad sequences to max_length
        truncation=True,       # Truncate sequences longer than max_length
        max_length=128         # Adjust max_length (128 is sufficient for SST-2)
    )

# Apply tokenization to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Rename the "label" column to "labels" (required for Hugging Face Trainer)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set format for PyTorch tensors
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [35]:
training_args = TrainingArguments(
    output_dir='./results_2',  # Updated directory name for SST-2 outputs
    num_train_epochs=5,  # Number of epochs for training (3 is standard for SST-2)
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    learning_rate=5e-5,  # Learning rate for optimizer (standard for fine-tuning transformers)
    adam_beta1=0.9,  # Beta1 parameter for Adam optimizer
    adam_beta2=0.999,  # Beta2 parameter for Adam optimizer (adjusted to standard value)
    adam_epsilon=1e-8,  # Epsilon parameter for Adam optimizer (adjusted to standard value)
    dataloader_drop_last=True,  # Set to False for SST-2 (important for small batches)
    warmup_steps=500,  # Increased warmup steps for better stability (adjust based on total steps)
    weight_decay=0.01,  # Weight decay for regularization
    logging_dir='./logs_2',  # Updated directory name for SST-2 logs
    logging_steps=50,  # Log every 50 steps (adjust based on dataset size and training speed)
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Save model checkpoints at the end of each epoch
    save_total_limit=3,  # Limit the number of saved checkpoints
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model="accuracy",  # Use accuracy to determine the best model
    fp16=True,  # Enable mixed precision training for faster training (if supported by hardware)
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}
    
# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


/home/sjiang/braids_v3/pip-test/lib/python3.13/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [36]:
# For COLA
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.687900,0.702212,0.509259
2,0.690600,0.694621,0.509259
3,0.683700,0.695380,0.509259
4,0.687900,0.695404,0.509259
5,0.685700,0.697114,0.509259


TrainOutput(global_step=21045, training_loss=0.6885515277814513, metrics={'train_runtime': 8371.8393, 'train_samples_per_second': 40.224, 'train_steps_per_second': 2.514, 'total_flos': 0.0, 'train_loss': 0.6885515277814513, 'epoch': 5.0})

In [37]:
trainer.evaluate()

{'eval_loss': 0.7022122740745544,
 'eval_accuracy': 0.5092592835426331,
 'eval_runtime': 5.797,
 'eval_samples_per_second': 150.423,
 'eval_steps_per_second': 9.488,
 'epoch': 5.0}

In [11]:
# For COLA
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.499000,0.572413,0.717593


TrainOutput(global_step=4209, training_loss=0.5911007991282327, metrics={'train_runtime': 1676.2664, 'train_samples_per_second': 40.178, 'train_steps_per_second': 2.511, 'total_flos': 0.0, 'train_loss': 0.5911007991282327, 'epoch': 1.0})

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.680200,0.696778,0.509174


TrainOutput(global_step=8418, training_loss=0.6892069759717829, metrics={'train_runtime': 3314.2258, 'train_samples_per_second': 20.321, 'train_steps_per_second': 2.54, 'total_flos': 0.0, 'train_loss': 0.6892069759717829, 'epoch': 1.0})

In [20]:
trainer.evaluate()

{'eval_loss': 0.6965716481208801,
 'eval_accuracy': 0.5092592835426331,
 'eval_runtime': 7.372,
 'eval_samples_per_second': 118.285,
 'eval_steps_per_second': 7.461,
 'epoch': 1.0}

In [ ]:
training_parallel = MyBertForSequenceClassification(model_parallel)

In [ ]:
# Load dataset
dataset = load_dataset('glue', 'mrpc')

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"], 
        examples["sentence2"], 
        padding="max_length", 
        truncation=True,
        max_length=224
    )
    
tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    adam_beta1=0.9,
    adam_beta2=0.988,
    adam_epsilon=1e-8,
    dataloader_drop_last=True,
    warmup_steps=5,
    weight_decay=1e-4,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
)

# Initialize the Trainer
trainer = Trainer(
    model=training_parallel,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)


In [ ]:
# For MRPC
trainer.train()